# 03 · Select jointly regularized predictors and controls


Direct-v3 fits one deterministic model per outer fold and arm. It does not repeat
the same solution under three seed labels. Legacy-v1 and direct-v2 keep their
historical stochastic heads and loading paths. All model choices are confined to
inner source validation; outer-test scores never choose a penalty or checkpoint.

**Run this notebook independently in a fresh kernel.** The default
`teach` mode uses small generated examples. Set `FI_TUTORIAL_MODE=inspect`
and `FI_RUN_ROOT` before starting the kernel to read saved artifacts.
Set `FI_TUTORIAL_MODE=execute` with an explicit `FI_RUN_ROOT` to run the
production stages below. Execute notebooks **00 → 04** in order for the
full Experiment 0; each uses a fresh kernel and the same run directory.
Use the [notebook HAIC launchers](../../../slurm/future-innovation/NOTEBOOKS.md)
for scheduled execution. Inspection remains read-only. An absent local
file says nothing about the current state of a remote HAIC job.

[Study overview](../../../docs/studies/future-innovation/README.md) ·
[Historical direct-v2 specification](../../../docs/studies/future-innovation/direct-gate-protocol.md) ·
[Calibrated direct-v3 specification](../../../docs/studies/future-innovation/direct-v3-repair-protocol.md)

In [ ]:
from pathlib import Path
import os
import sys
from time import perf_counter

started = perf_counter()
override = os.environ.get("GAVD6_ROOT")
if override:
    candidates = [Path(override).expanduser().resolve()]
else:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / "gavd6", base / "experiments/sjepa/gavd6"))
PROJECT_ROOT = next((p for p in candidates if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the checkout containing src/gavd6_sjepa.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats
get_ipython().run_line_magic("matplotlib", "inline")
set_matplotlib_formats("svg", "png")
plt.rcParams.update({"figure.figsize": (8, 3), "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 11})

from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import (
    artifact_inventory, inspect_report, read_optional_table, inspection_audit_path,
)

# Use "execute" for real stages or "inspect" for saved artifacts.
# Relative paths resolve from GAVD6_ROOT. Execution requires an explicit run root.
MODE = os.environ.get("FI_TUTORIAL_MODE", "teach")
if MODE not in {"teach", "inspect", "execute"}:
    raise ValueError("FI_TUTORIAL_MODE must be teach, inspect, or execute.")
if MODE == "execute" and not os.environ.get("FI_RUN_ROOT"):
    raise ValueError("Set FI_RUN_ROOT explicitly before executing real experiment stages.")
RUN_ROOT = Path(os.environ.get("FI_RUN_ROOT", "outputs/future-innovation-direct-v3-dev-20260911")).expanduser()
if not RUN_ROOT.is_absolute():
    RUN_ROOT = PROJECT_ROOT / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("Teaching examples only; no empirical gait findings." if MODE == "teach"
      else f"{MODE.upper()} mode: {RUN_ROOT}")
if MODE == "execute":
    from gavd6_sjepa.research_directions.future_innovation.fi_notebook_workflow import (
        initialize_from_environment, run_stage, build_notebook_report, finish_notebook_report,
        attempt_stage, require_stage_success,
    )
    if (RUN_ROOT / "config/run-contract.json").is_file():
        import json
        saved_run = json.loads((RUN_ROOT / "config/run-contract.json").read_text())
        print("Frozen protocol:", saved_run.get("protocol", "legacy-v1"),
              "— gate clips:", saved_run.get("cohort_size"))
        if saved_run.get("protocol", "legacy-v1") == "legacy-v1":
            print("This run retains legacy selectivity gates. Use a new run root for direct-v2.")

## Execute this stage

Run five outer folds, all frozen arms and the complete frozen inner grid. Direct-v3 has one deterministic fit; historical protocols retain three seeds. With FI_NOTEBOOK_FOLD set, run just that outer fold (all seeds and arms); HAIC supplies five CPU array tasks. Without it, run all folds sequentially. A verified audit rejection skips fitting and records a blocked outcome. No teaching settings enter this branch.

This cell runs only in `execute` mode. Each command uses this kernel's Python and the existing production CLI; stage logs are retained alongside the executed notebook.

In [ ]:
if MODE == "execute":
    fold = os.environ.get("FI_NOTEBOOK_FOLD")
    if fold is not None and fold not in {"0", "1", "2", "3", "4"}:
        raise ValueError("FI_NOTEBOOK_FOLD must be 0–4; unset it to run all five folds.")
    options = [] if fold is None else ["--outer-fold", fold]
    stage_error = attempt_stage("run-gate", RUN_ROOT, "--device", "cpu", *options)

Four ordered bins `[0,8)`, `[8,16)`, `[16,24)`, `[24,32)` preserve temporal position.
Each of 33 joints contributes mean valid x/y, valid adjacent-frame velocity x/y,
confidence, frame support and transition support: 924 fixed features. Missing
observations are imputed from training means only. Unsupported or near-constant
training columns become exact zeros in every partition, with their positions
retained. Target scaling is a separate operation and keeps target variation.

| Arm | Transformation before feature construction |
|---|---|
| Real | Original coordinate/confidence/validity history |
| Time shuffle | Permute four-frame blocks, moving all channels together |
| Clip mismatch | Different-source, context-matched donor inside the current partition |
| No skeleton | Zero x/y/confidence; retain the original time-varying validity |

All arms receive 36 pairs from the same positive RGB/skeleton penalty grids plus
the same selected RGB-only baseline. Inner loss pools source-weighted error sums
and weight totals. Ties within the frozen numerical tolerance prefer baseline;
other ties prefer stronger penalties. Failed candidates stay visible and prevent
a complete scientific result. The exact fallback can win every comparison.

In [ ]:
if MODE=='teach':
    from gavd6_sjepa.research_directions.future_innovation.fi_joint_calibration import exact_calibration
    from gavd6_sjepa.research_directions.future_innovation.fi_joint_models import JointRidge
    demo=exact_calibration()
    display(pd.DataFrame([demo]))
    rng=np.random.default_rng(41)
    x=rng.normal(size=(20,4)); s=rng.normal(size=(20,2)); y=x[:,:1]+s[:,:1]
    joint=JointRidge.fit(x,s,y,np.ones(20),10.,10.)
    display(pd.DataFrame({'illustrative_target':y[:,0],'training_prediction':joint.predict(x,s)[:,0]}))
    print('Synthetic training illustration only; real calibration uses complete source-held selection.')

The historical all-candidates-loss problem is visible in each inner partition,
not just its pooled score. Large losses should stay in the numerical table even
when a logarithmic plot is easier to read. Direct-v3 train and validation losses
come from the same closed-form fit within each inner partition. Historical
neural training loss was recorded before an update and validation afterward;
new timing labels describe that difference. It was not the headline R² cause.

Inspect supported feature counts alongside nominal dimensions. Baseline-only
uses no skeleton coefficients. A joint candidate that wins inner validation can
still lose on unseen sources; fallback eligibility does not guarantee outer gain.

In [ ]:
if MODE!='teach':
    from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import prediction_fit_tables
    from gavd6_sjepa.research_directions.future_innovation.fi_contracts import read_json
    path=RUN_ROOT/'config/model-contract.json'
    if path.is_file(): display(read_json(path))
    tables=prediction_fit_tables(RUN_ROOT,os.environ.get('FI_NOTEBOOK_FOLD'))
    for name,table in tables.items():
        if not table.empty:
            print(name, f'({len(table)} records; full machine-readable records under models/fold-*/.)')
            with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.precision',10):
                display(table)
    candidates=tables['candidates']
    if not candidates.empty:
        selected=tables['selected']
        print('Baseline fallback frequency:', (selected.selected_type=='baseline_only').sum(), '/',len(selected))
        fig,ax=plt.subplots(figsize=(9,3))
        for arm,rows in candidates.groupby('arm'):
            ax.scatter(np.arange(len(rows)), rows.pooled_loss, s=12, alpha=.55, label=arm)
        ax.set(yscale='log',xlabel='Candidate record within arm',ylabel='Pooled inner MSE',title='All candidate losses; numeric values retained above')
        ax.legend(); plt.show()
    elif not tables['historical_inner_fits'].empty:
        history=tables['historical_inner_fits']
        fig,ax=plt.subplots(figsize=(9,3))
        ax.scatter(np.arange(len(history)),history.validation_squared_error/history.validation_weight,s=8)
        ax.set(yscale='log',xlabel='Historical inner fit record',ylabel='Validation MSE'); plt.show()
    display(artifact_inventory(RUN_ROOT))

In [ ]:
if MODE == "execute":
    require_stage_success(stage_error)

## What this step establishes

Every selected model has an inner-validation reason, a fitted preprocessing record and an explicit type. Next reconstruct its held-out predictions and assess the paired effect and uncertainty.

Continue with [04_results_and_next_decision.ipynb](04_results_and_next_decision.ipynb).

In [ ]:
print(f"Notebook elapsed time: {perf_counter() - started:.2f} seconds ({MODE} mode).")